<a href="https://colab.research.google.com/github/pakkei1212/smu-llm-group-project/blob/deepeval/notebooks/medical-rag-eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Packages

In [ ]:
%pip install deepeval==3.9.1 accelerate==1.13.0 transformers==5.3.0 langchain-mistralai

# Imports

In [ ]:
import torch
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models.base_model import DeepEvalBaseLLM
import asyncio
import csv
import os
import json
from tqdm import tqdm
from google.colab import userdata
from google.colab import drive

drive.mount('/content/drive')

# Load Test Cases

In [ ]:
base_dir = "/content/drive/MyDrive/RAG Outputs GPT"

test_cases = {}

for filename in os.listdir(base_dir):
  if 'rag2' not in filename:
    continue

  test_cases[filename] = []

  with open(f"{base_dir}/{filename}", "r") as csv_file:
    reader = csv.DictReader(csv_file)

    for row in reader:
      if "input" in row and "actual_output" in row and "retrieval_context" in row:
          test_cases[filename].append(
              LLMTestCase(
                  input=row["input"].strip(),
                  actual_output=row["actual_output"].strip(),
                  retrieval_context=row["retrieval_context"].split("\n\n---\n\n")
              )
          )

    print(f"Loaded test cases from {filename}")

# Evaluate

### HuggingFace

In [ ]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

class EvalModel(DeepEvalBaseLLM):
    def __init__(self, model, tokenizer):
        self.pipe = None
        self.load_model(model, tokenizer)

    def load_model(self, model, tokenizer):
        self.pipe = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
        )
        return self.pipe

    def _generate(self, prompt: str) -> str:
        messages = [{"role": "user", "content": prompt}]

        outputs = self.pipe(
            messages,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=True,
            pad_token_id=self.pipe.tokenizer.eos_token_id
        )

        return outputs[0]["generated_text"][-1]["content"].strip()

    def generate(self, prompt: str, **kwargs) -> str:
        return self._generate(prompt)

    async def a_generate(self, prompt: str, **kwargs) -> str:
        loop = asyncio.get_running_loop()
        return await loop.run_in_executor(None, self._generate, prompt)

    def generate_raw_response(self, prompt: str, **kwargs) -> str:
        response = self._generate(prompt)
        return (response, 0.0)

    async def a_generate_raw_response(self, prompt: str, **kwargs) -> str:
        loop = asyncio.get_running_loop()
        response = await loop.run_in_executor(None, self._generate, prompt)
        return (response, 0.0)

    def get_model_name(self) -> str:
        return "Qwen3.5"

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3.5-4B",
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=False
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B")

evaluator = EvalModel(model, tokenizer)

### Mistral

In [ ]:
from langchain_mistralai import ChatMistralAI

class EvalModel(DeepEvalBaseLLM):
    def __init__(self, model):
        self.model = model

    def load_model(self):
        return self.model

    def _generate(self, prompt: str) -> str:
        output = self.load_model().invoke(prompt).content

        return output

    def generate(self, prompt: str, **kwargs) -> str:
        return self._generate(prompt)

    async def a_generate(self, prompt: str, **kwargs) -> str:
        loop = asyncio.get_running_loop()
        return await loop.run_in_executor(None, self._generate, prompt)

    def generate_raw_response(self, prompt: str, **kwargs) -> str:
        response = self._generate(prompt)
        return (response, 0.0)

    async def a_generate_raw_response(self, prompt: str, **kwargs) -> str:
        loop = asyncio.get_running_loop()
        response = await loop.run_in_executor(None, self._generate, prompt)
        return (response, 0.0)

    def get_model_name(self) -> str:
        return "Mistral"

mistral = ChatMistralAI(
    model="mistral-small-2506",
    temperature=0,
    max_retries=2,
    api_key=userdata.get('MISTRAL_API_KEY')
)

evaluator = EvalModel(mistral)

### Gemini

In [ ]:
from deepeval.models import GeminiModel
from google.colab import userdata

evaluator = GeminiModel(
    model="gemini-3.1-flash-lite-preview",
    # model="gemini-2.5-flash-lite",
    # model="gemma-3-27b-it",
    # model="gemini-2.5-pro",
    # model="gemini-3.1-flash-lite-preview",
    # model="gemini-2.0-flash",
    api_key=userdata.get('GEMINI_API_KEY'),
    temperature=0
)

### OpenAI

In [ ]:
from deepeval.models import GPTModel
from google.colab import userdata

evaluator = GPTModel(
    api_key=userdata.get('OPENAI_API_KEY'),
    model="gpt-4.1",
    temperature=0
)

## DeepEval

In [ ]:
# Increase per-task timeout (in seconds)
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "300"

# Increase buffer time for gathering results
os.environ["DEEPEVAL_TASK_GATHER_BUFFER_SECONDS_OVERRIDE"] = "60"

def evaluate_test_cases(testcases):
  factuality = GEval(
      name="Factuality & Grounding",
      criteria="Determine if actual_output is factually correct and fully grounded in retrieval_context.",
      evaluation_steps=[
          "List all factual claims made in the actual_output.",
          "Verify each claim is directly supported by retrieval_context.",
          "Penalize hallucinations, speculation, or unsupported details.",
          "Score 1-10: 10=perfectly factual/grounded, 1=fully hallucinated."
      ],
      evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.RETRIEVAL_CONTEXT],
      model=evaluator,
      threshold=0.7
  )

  recall = GEval(
      name="Recall",
      criteria="Assess if retrieval_context contains all information needed to fully answer the input query.",
      evaluation_steps=[
          "Identify key facts required to answer the input query.",
          "Check if retrieval_context includes these key facts.",
          "Penalize missing critical information needed for complete answer.",
          "Score 1-10: 10=context fully supports ideal answer, 1=critical info missing."
      ],
      evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.RETRIEVAL_CONTEXT],
      model=evaluator,
      threshold=0.7
  )

  f1_score = GEval(
      name="F1 Score",
      criteria="Evaluate the balance between precision (no extra info) and recall (all necessary info) in the actual_output relative to the provided context.",
      evaluation_steps=[
          "Identify necessary components of an answer based on the context.",
          "Check if actual_output contains unnecessary or incorrect information (Precision).",
          "Check if actual_output contains all necessary information found in the context (Recall).",
          "Calculate a score representing the balance (F1) of these two factors."
      ],
      evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.RETRIEVAL_CONTEXT],
      model=evaluator,
      threshold=0.7
  )

  return evaluate(
      test_cases=testcases,
      async_config=AsyncConfig(run_async=True, throttle_value=5, max_concurrent=3),
      metrics=[f1_score, recall, factuality]
  ).model_dump()

for filename, file_test_cases in tqdm(iterable=test_cases.items(), total=len(test_cases.keys())):
  results = evaluate_test_cases(file_test_cases)

  json.dump(results, open(f"{base_dir}/{filename}.json", "w"))

In [ ]:
# print("\n=== RESULTS ===")
# print(f"{'Case':<4} {'Fact':<6} {'Rec':<6} {'Pass':<6} {'Reason':<50}")
# print("-" * 90)

# for idx, result in enumerate(results.test_results):
#   fact_score = result.metrics_data[0].score  # Factuality
#   recall_score = result.metrics_data[1].score  # Recall
#   fact_reason = result.metrics_data[0].reason[:45]

#   status = "Y" if fact_score >= 0.7 and recall_score >= 0.7 else "N"

#   print(f"{idx:<4} {fact_score:<6.3f} {recall_score:<6.3f} {status:<6} {fact_reason}")

# fact_pass = sum(1 for r in results.test_results if r.metrics_data[0].score >= 0.7) / len(results.test_results)
# recall_pass = sum(1 for r in results.test_results if r.metrics_data[1].score >= 0.7) / len(results.test_results)
# both_pass = sum(1 for r in results.test_results if r.metrics_data[0].score >= 0.7 and r.metrics_data[1].score >= 0.7) / len(results.test_results)

# print(f"\nSummary:")
# print(f"Factuality Pass Rate: {fact_pass:.1%}")
# print(f"Recall Pass Rate: {recall_pass:.1%}")
# print(f"Both Pass Rate: {both_pass:.1%}")